# Этап R.0: есть ли адрес числа подлежащего в моделях Pythia

Проверяется, есть ли в остаточном потоке на последнем токене одномерный адрес числа подлежащего, который
воспроизводит собственный ответ модели на смену числа (R² подмены, эталон — DAS). Модели — PolyPythias
14m–160m, seeds 1–9, пилот — seed 0 и шесть дополнительных моделей 160m. Протокол целиком записан в начале кода
(часть 2.1).

**Порядок работы**

| раздел | что делает | время на T4 |
|---|---|---|
| 1 | среда и папка на Google Диске | 1 мин |
| 2 | код этапа и сборка файла с проверкой sha256 | 1 мин |
| 3–5 | пробный запуск, замер, пилот — **уже сделаны**, выключены галочками | — |
| 6 | что прислать после пилота (сделано) | — |
| 7 | **основной прогон** | 36 моделей ≈ 1,5 ч, затем 410m ≈ 2 ч |
| 8 | сводка основного прогона и скачивание | — |

**Как запускать.** Среда выполнения → Сменить среду выполнения → **T4 GPU**. Затем Среда выполнения →
**Выполнить все**: выполнятся разделы 1, 2 и 7, а 3–5 пропустятся, пока их галочки выключены.

Если сессия оборвётся: снова выполнить разделы 1 и 2, затем прерванную ячейку. Пилот и основной прогон
продолжат с места остановки — готовые модели сохраняются в папке на Диске.

Код собран из файла `scripts/pythia_address.py`, sha256 `6c0606476a6e0c1f…`.

## 1. Среда и папка

In [ ]:
#@title 1.1 Видеокарта и версии
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch, transformers, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| transformers", transformers.__version__, "| numpy", numpy.__version__)
assert torch.cuda.is_available(), "нет видеокарты: Среда выполнения -> Сменить среду выполнения -> T4 GPU"


In [ ]:
#@title 1.2 Папка на Google Диске (результаты переживут обрыв сессии)
FOLDER = "/content/drive/MyDrive/R0"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(FOLDER, exist_ok=True)
%cd {FOLDER}


## 2. Код этапа R.0

Код — это файл `scripts/pythia_address.py` из репозитория, разбитый на 11 частей. Каждая ячейка хранит свою часть
как текст; чтобы посмотреть код, нажмите «Показать код». Ячейка 2.12 собирает файл и проверяет его sha256: если
хоть один символ изменён, она остановится, и прогон будет недействителен. **Ничего в этих ячейках не менять.**

In [ ]:
#@title 2.0 Подготовка
PARTS = {}


In [ ]:
#@title 2.1 Протокол этапа (docstring) и импорты
PARTS[0] = r'''#!/usr/bin/env python
"""Stage R.0: does the number of the subject have an address inside pretrained language models?

PROTOCOL. Committed to the repository before the run, once.
---------------------------------------------------------------------------
Role.       Stage 2.0 repeated on models trained by other people: a check of the
            instrument, not of the interpreter. DAS (Geiger, Wu, Potts, Icard and Goodman,
            CLeaR 2024) gives the reference address inside one model. The interpreter is
            tested in stage R.1, under its own protocol, after this result.
            Closest prior work: CausalGym (Arora, Jurafsky and Potts, ACL 2024) ran DAS
            and other methods on Pythia models, including subject-verb agreement
            (agr_sv_num_subj-relc, agr_sv_num_obj-relc, agr_sv_num_pp). They used the
            residual stream after a layer at the last token of a region, one-dimensional
            interventions and a log odds-ratio. Their main text does not map where
            agreement sits, so the layer here is chosen by the pilot below. The metric here
            is stricter: interchange R^2 against the model's own counterfactual output.
Models.     PolyPythias (van der Wal et al., ICLR 2025): EleutherAI/pythia-{size}-seed{1..9},
            sizes 14m, 31m, 70m and 160m. The revisions are pinned in REVISIONS below. Each
            seed changes the initialisation and the batch composition; the corpus (the
            standard Pile) and the hyper-parameters are shared.
            Pilot only, never in the check: seed 0 of every size (the original
            EleutherAI/pythia-{size}) and pythia-160m-{data,weight}-seed{1,2,3}.
            pythia-410m: descriptive only, and only if the time measurement allows it (see
            'Precision').
Task.       Subject-verb number agreement. Three templates over fixed token positions, every
            word one token of the GPT-NeoX tokenizer (checked by the script):
              rc_obj  "The {mod} {head} that the {people} {verb}"   (section 3.7, template 1)
              rc_subj "The {mod} {head} that {verb} the {people}"   (section 3.7, template 2)
              pp      "The {mod} {head} {prep} the {noun}"          (as agr_sv_num_pp)
            The verb of the relative clause agrees with its own subject. B(x) is
            logit(" are") - logit(" is") at the last position.
Variables.  SUBJ, the number of the subject: carriers head (and in rc_subj the clause's
            verb, which agrees with it). ATTR, the number of the attractor: people / noun
            (and in rc_obj the clause's verb). At the last token ATTR is carried by the last
            word itself in every template; its address there is close to the input, so it
            is read separately and its result says so.
Pairs.      Minimal pairs: x_cf is the base sentence with the variable's number taken from
            the source, the base's words kept (the agreeing verb follows). Base and source
            are drawn independently from the same template, the template uniformly. Per
            variable: 4096 training, 1024 validation and 2048 test pairs, numpy
            default_rng([PAIR_STREAM, variable, split]); the same sentences for every model.
            A pair whose base and source have the same number is not informative.
Points.     The residual stream at index r (r = 0 the embeddings, r = j after block j) at the
            last position, k = 1 (the variable is binary). Two main points per size:
              last     r* = the layer chosen by the rule written before the pilot;
              arrival  the first layer whose mean pilot DAS R^2 is >= 0.50: where the number
                       of the subject first arrives at the last token. This criterion was
                       chosen AFTER the pilot, on the pilot models only (seed 0 and the six
                       160m models); the check seeds 1-9 were not used for it.
            Both, from 'Pilot': last 14m 6, 31m 6, 70m 6, 160m 12, 410m 24; arrival 14m 6,
            31m 5, 70m 5, 160m 7, 410m 14. In 14m the two points coincide (decided before
            the run): its arrival values are its last-layer values, not a second fit.
            The subject's own position is a descriptive diagnostic at r = 1. The draft had it
            at the same r as the main point; at r* = the last layer the subject position has no
            path to the output, so every reading there is identically zero (a test shows it).
Metric.     Interchange R^2 = 1 - sum (B(h') - B(x_cf))^2 / sum (B(x_cf) - B(x_b))^2 over
            the test pairs, h' = h_b + ((h_s - h_b) . q) q, as in stage 2.0. A (model,
            variable) with fewer than 50 informative test pairs (|B(x_cf) - B(x_b)| >= 0.10
            sigma_B, sigma_B the standard deviation of B over 256 probe sentences) is left
            out of every mean.
DAS.        Stage 2.0's recipe: q = w / |w|; loss mean(((B(h') - B(x_cf)) / sigma_B)^2);
            Adam, lr 1e-2, 500 steps, batch 256, 3 restarts, the restart with the best
            validation R^2. The result is the test R^2.
Formulas with zero pairs, k = 1, from 4096 sentences of the model's own distribution
            (numpy default_rng([FORMULA_STREAM])), one forward and one backward pass each:
            without labels (the interpreter of R.1 gets the same signals):
              pca          the leading principal component of the residuals;
              first_order  the mean over sentences of J . de, the derivative of the
                           residual along de = e(plural form) - e(singular form) at the
                           variable's carriers (a Jacobian-vector product, label-free:
                           the same direction for every sentence);
              gradient     the mean of dB/dh;
            WITH LABELS of the variable's value on the checked model, which the interpreter
            of R.1 does not get (author's decision, 2026-10-08):
              diffmean     mean residual of plural minus singular sentences;
              lda          (Sigma_w + lambda I)^-1 (mu_1 - mu_0), lambda = 1e-3 tr(Sigma_w)/d;
              probe        L2 logistic regression (1e-3) on residuals scaled by one number;
            and random: the mean over 10 random unit vectors
            (numpy default_rng([RANDOM_STREAM, size, seed, variable])).
Unit.       The seed (1..9). A unit's value is the mean over the four sizes 14m-160m of the
            model's test R^2. With 9 units the smallest two-sided sign-flip p is
            2/512 = 0.0039, so a Holm family of m comparisons can reject only if m <= 12;
            per-size results are descriptive for that reason.
Read-out (a), existence. Family: DAS vs random for SUBJ at the last layer, SUBJ at the
            arrival layer and ATTR at the last layer; exact two-sided sign-flip test over the
            9 units, Holm over these 3. Per (variable, point): CONFIRMED if the comparison
            rejects with a positive difference AND the mean DAS R^2 over the units is
            >= 0.50; NOT CONFIRMED otherwise.
Read-out (b), is the address analytic. Per point, DAS vs each of the six formulas for
            SUBJ, Holm over the six of that point; the two points are reported separately.
            Per formula, as in the check of stage 2.0: ANALYTIC if the 95% bootstrap
            interval of the mean difference DAS - formula lies inside [-0.05, +0.05]; DAS
            HIGHER otherwise if the comparison rejects with a positive difference; FORMULA
            HIGHER if it rejects with a negative one; UNDETERMINED otherwise. Per point:
            ADDRESS ANALYTIC if any formula is ANALYTIC or FORMULA HIGHER; DAS HIGHER THAN
            EVERY FORMULA if all six are DAS HIGHER; MIXED otherwise. The labelled formulas
            are marked. Whatever (b) says, R.1 is run (author's decision); an analytic address
            is reported as a result of its own.
Stop rule.  R.1 is built at each point where (a) is CONFIRMED for SUBJ. If at neither,
            neither R.1 nor part 3 on real models is built.
For R.1, decided by the author on 2026-10-08 and recorded here before R.0's run:
            both points enter R.1's main rule. Per point: (a) the interpreter's R^2 is
            significantly above the strongest formula WITHOUT labels; (b) it is not worse
            than the strongest formula WITH labels by more than 0.05 (TOST). Family: 2 points
            x 2 conditions = 4 comparisons, Holm; each point reported separately (2 if R.0
            confirms one point only). The strongest formulas, chosen on the pilot (see
            'Pilot'): last layer gradient / probe; arrival layer first_order / diffmean.
            R.1's own protocol is written after this result.
Descriptive, outside the families: every size; every template; decision IIA; the
            subject-position diagnostic at r = 1; ATTR's formulas; 410m; the pilot models.
Statistics. 95% percentile bootstrap intervals over the units (10 000 resamples, numpy
            default_rng(0)).
Controls.   On every model, before any fitting; a failure stops that model's run:
            (1) this file's forward pass equals the transformers forward;
            (2) continuing from the cached residual reproduces B(x);
            (3) replacing the whole residual vector at (r, p) by the source's equals the
                same replacement made inside a full forward pass;
            (1)-(3) are computed in float64 (the weights go fp32 -> fp64 -> fp32, which is
                exact), tolerance |difference| <= 1e-8 (1 + |B|). In fp32 both this forward
                pass and the transformers one move B by about 1e-3 on pythia-14m (found on
                the first smoke run, where an fp32 control at 1e-4 stopped); that rounding is
                recorded per model as a descriptive number;
            (4) every minimal pair differs from its base only at the variable's carriers.
Precision.  fp32 for 14m-160m, TF32 off. Determinism flags are set and recorded:
            torch.use_deterministic_algorithms(True, warn_only=True),
            CUBLAS_WORKSPACE_CONFIG=:4096:8, cudnn.deterministic, no cudnn benchmark.
            410m: --time measures a DAS step on the Colab GPU. Rule written before the
            measurement: 410m is run, descriptively and with fp16 autocast, if the measured
            projection of its DAS time for 9 seeds in fp16 (2 variables and the diagnostic,
            3 restarts x 500 steps each) is at most 3 hours; otherwise it is dropped.
            Measured (results/pythia_address_time.json, Tesla T4, one DAS step at batch 256,
            r = L/2): 160m 0.073 s fp32, 0.091 s fp16; 410m 0.217 s fp32, 0.169 s fp16, peak
            GPU memory 6.5 GB. Projection for 410m: 1.91 h in fp16 (2.45 h in fp32). By the
            rule 410m is run, descriptively, in fp16; it may be finished in a second session.
Provenance. Every per-model file and the final file record this script's sha256, torch,
            transformers, numpy and python versions, CUDA, the GPU model, the precision
            mode, the determinism flags and the pinned model revisions. The run is valid only
            if the recorded sha256 equals this file at the protocol commit; `--verify`
            checks that locally before the result is committed.
Amendment 1, 2026-10-09, after the 36 confirmatory models were computed and before any result
            was aggregated. The run stopped at the first descriptive 410m model: under fp16
            autocast the model's outputs (fp16) were written into an fp32 array ("expected scalar
            type Half but found Float"). The time measurement had not met this, because it built
            its pairs outside autocast. Fixed by casting the outputs to fp32, with a test that runs
            the half-precision path. So that the confirmatory part is not recomputed, a per-model
            file of 14m-160m is reused if it was written by the script of the protocol commit
            (PROTOCOL_COMMIT, PROTOCOL_SHA256). The confirmatory result is written to
            pythia_address.json as soon as the 36 models are done, and 410m to its own file
            pythia_address_410m.json. Nothing in the confirmatory computation, the read-outs or
            the families changes (git diff of the two commits); --verify checks the hashes.
Pilot.      On the pilot models only, with a lighter recipe (1024 / 512 pairs, 300 steps,
            1 restart): DAS validation R^2 for SUBJ at the last position at every r (410m every
            second r) and the formulas at every r. Rule written before the pilot: per size, r*
            is the r with the highest mean validation DAS R^2 over that size's pilot models
            (ties: the smaller r); the strongest formula without labels and the strongest with
            labels are the ones with the highest validation R^2 at r*, averaged over the pilot
            models of a size and then over the sizes 14m-160m.
            Run on Colab (Tesla T4, torch 2.11.0+cu130, transformers 5.18.0) with this script
            at commit f808396 (sha256 97e22080...). The first Colab smoke had stopped at
            loading, because transformers 5 renamed the output head; that was fixed in
            f808396 before the pilot. Files: results/pythia_address_pilot.json and
            results/pythia_address_time.json. Every float64 control passed (worst 1.6e-11).
            `--pilot-summary results/pythia_address_pilot.json` reproduces everything below,
            and a test checks that the constants of this file agree with it.
            Mean validation DAS R^2 at the last token by layer r:
              14m  (1 model)  0:.05 1:.05 2:.06 3:.04 4:.05 5:.04 6:.82
              31m  (1)        0:.07 1:.05 2:.07 3:.09 4:.19 5:.53 6:.82
              70m  (1)        0:.08 1:.10 2:.06 3:.20 4:.23 5:.65 6:.86
              160m (7)        0:.05 1:.06 2:.07 3:.05 4:.12 5:.18 6:.24 7:.66 8:.72 9:.70
                              10:.88 11:.89 12:.90
              410m (1)        0:.05 2:.10 4:.06 6:.09 8:.10 10:.17 12:.25 14:.74 16:.76
                              18:.81 20:.82 22:.83 24:.92
            The rule chose the last layer in every size: r* = 6, 6, 6, 12, 24. That is the
            state just before the unembedding. The author then added the arrival layer as a
            second main point (criterion above, chosen after seeing this table).
            Validation R^2 at the two points (DAS | gradient, first_order | probe, diffmean):
              last     14m .82 | .23 .37 | .61 .30     arrival  14m (the same point)
                       31m .82 | .43 .51 | .57 .41              31m r5 .53 | -.11 .27 | .03 .29
                       70m .86 | .64 .47 | .49 .61              70m r5 .65 | .33 .45 | .25 .45
                       160m .90 | .77 .45 | .76 .47             160m r7 .66 | .44 .38 | .45 .47
                       410m .92 | .79 .42 | .83 .70             410m r14 .74 | .62 .18 | .62 .64
            Pooled over 14m-160m, last layer: DAS .851; without labels gradient .517,
            first_order .450, pca -.013; with labels probe .608, diffmean .447, lda .085.
            Arrival layer: DAS .664; without labels first_order .367, gradient .220,
            pca -.002; with labels diffmean .379, probe .337, lda .045.
            Strongest for R.1: last layer gradient / probe (by the rule written before the
            pilot); arrival layer first_order / diffmean (by the same rule applied at the
            arrival layer).
---------------------------------------------------------------------------
Colab (T4):
    !python pythia_address.py --smoke                   # minutes, writes nothing
    !python pythia_address.py --time                    # the 410m time measurement
    !python pythia_address.py --pilot                   # writes pythia_address_pilot.json
    !python pythia_address.py --pilot-summary pythia_address_pilot.json   # the choices from it
    !python pythia_address.py --out R0                  # the run: R0/*.json, then pythia_address.json
Locally, before committing a result:
    python scripts/pythia_address.py --verify pythia_address.json
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from itertools import product

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
import torch.nn.functional as F

'''


In [ ]:
#@title 2.2 Константы, потоки случайных чисел, закреплённые ревизии моделей
PARTS[1] = r'''# ------------------------------------------------------------------ constants
SIZES = ("14m", "31m", "70m", "160m")
SIZE_410 = "410m"
SIZE_ID = {"14m": 0, "31m": 1, "70m": 2, "160m": 3, "410m": 4}
CHECK_SEEDS = tuple(range(1, 10))
PILOT_MODELS = tuple(f"pythia-{s}" for s in SIZES + (SIZE_410,)) + tuple(
    f"pythia-160m-{kind}-seed{i}" for kind in ("data", "weight") for i in (1, 2, 3))
VARIABLES = ("SUBJ", "ATTR")
VAR_ID = {"SUBJ": 0, "ATTR": 1}
PAIR_STREAM, FORMULA_STREAM, PROBE_STREAM, RANDOM_STREAM, TORCH_STREAM = 3030, 3031, 3032, 3033, 3034
FULL = dict(n_train=4096, n_val=1024, n_test=2048, steps=500, restarts=3, batch=256, lr=1e-2)
PILOT = dict(n_train=1024, n_val=512, n_test=0, steps=300, restarts=1, batch=256, lr=1e-2)
SMOKE = dict(n_train=64, n_val=32, n_test=32, steps=3, restarts=1, batch=32, lr=1e-2, n_formula=64)
N_FORMULA, N_PROBE, N_RANDOM = 4096, 256, 10
TOL_INFORMATIVE, MIN_INFORMATIVE, BAR, MARGIN = 0.10, 50, 0.50, 0.05
N_BOOT = 10_000
CONTROL_TOL = 1e-8          # |difference| / (1 + |B|), in float64
UNLABELLED = ("pca", "first_order", "gradient")
LABELLED = ("diffmean", "lda", "probe")
FORMULAS = UNLABELLED + LABELLED
# fixed after the pilot, before the commit (see 'Pilot'); pilot_summary() derives them from
# results/pythia_address_pilot.json and a test checks that they agree
R_STAR = {"14m": 6, "31m": 6, "70m": 6, "160m": 12, "410m": 24}     # last layer: rule written before the pilot
R_ARRIVAL = {"14m": 6, "31m": 5, "70m": 5, "160m": 7, "410m": 14}   # arrival: criterion chosen AFTER the pilot
ARRIVAL_BAR = 0.50
R_DIAG = 1                                                          # subject-position diagnostic
STRONGEST = {"last": {"unlabelled": "gradient", "labelled": "probe"},
             "arrival": {"unlabelled": "first_order", "labelled": "diffmean"}}
WITH_410M = True                                                    # descriptive, fp16 (see 'Precision')
# Amendment 1: the 36 confirmatory models were computed by the script of the protocol commit
PROTOCOL_COMMIT = "5a57e39"
PROTOCOL_SHA256 = "c4f4a9dff16f45be682843418b46e5b7ee37a784fd14969fe6cc81aa3ee7f76a"
CONFIRMATORY_FILE, DESCRIPTIVE_410M_FILE = "pythia_address.json", "pythia_address_410m.json"
POINTS = {"last": "SUBJ", "arrival": "SUBJ_arrival"}

REVISIONS = {
    # pinned from the Hugging Face API on 2026-10-08; main = step143000
    "pythia-14m": {"sha": "cf967c0a9a04383db6f7b1108d86b2962634b4ac", "weights": "model.safetensors", "bytes": 28143920},
    "pythia-14m-seed1": {"sha": "0c929f682ef6f3c6582c3f0518713b53117225e2", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed2": {"sha": "d34e2feab87a7f8631f7335aac4d85aafbe47663", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed3": {"sha": "296092457f9e92450383421fc13c7601cdebba76", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed4": {"sha": "c1d2ccdcd54c661e5536c41e185fd97698e0f38d", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed5": {"sha": "b86476f2829441a386c32f330d45e06c79699db5", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed6": {"sha": "cb8e821ea1e0237339e2ea59cf2666cdf2e9b73b", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed7": {"sha": "7bafc17291b96a924536134ced88a045e5152a51", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed8": {"sha": "96bed32adeb238865e691de2168a138a513533fb", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed9": {"sha": "50e806022dda859254768ba4ceae24788b49aee0", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-31m": {"sha": "e556ace21b489575e94e9d50b6dad2fcc7419679", "weights": "model.safetensors", "bytes": 60997960},
    "pythia-31m-seed1": {"sha": "9d2c5119aa939b5abeef3157c94122fa69e8466a", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed2": {"sha": "7a2c0cfe7793936ce1596d1f5f14d00d1e29e7e5", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed3": {"sha": "007780274aabaef059324dfb91929e512fbe29ad", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed4": {"sha": "500f4c7ccfc248bcff33e6a9ede92af0f77268c4", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed5": {"sha": "057af9a8057a13713396963a38f272574f38ff83", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed6": {"sha": "285752bb715b22eb0a423c34f3d85799f0a491fd", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed7": {"sha": "4baadf03d6cdccbc04bc90a438676bca1e24e5a5", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed8": {"sha": "241aa615964e20fb411aea451a241c6d4fbab683", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed9": {"sha": "cf8491770336a746113d26ebd41961cbb83cd911", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-70m": {"sha": "a39f36b100fe8a5377810d56c3f4789b9c53ac42", "weights": "model.safetensors", "bytes": 166029852},
    "pythia-70m-seed1": {"sha": "b92214180642ad85f91ccbfffcbf118ebcf2529f", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed2": {"sha": "9df96128bfe65ea5e7f1543f0c8ee707932df286", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed3": {"sha": "3b039e52757be58fe0f769bec63cf50254853f88", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed4": {"sha": "f525bf61a55437bf9166a77571f8e0c054215600", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed5": {"sha": "e27a605ee68762a318956435821ef7ac23c21199", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed6": {"sha": "b4e7ddaab555b55d9043c210510795ebb0787be1", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed7": {"sha": "17e376beb51f5fbf51a47d1e9368fdaa343fe498", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed8": {"sha": "ecb4649b5096fed7e206bea3b8abd22192c0c601", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed9": {"sha": "373172e086d06a8a9a15816db498d099246f1409", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-160m": {"sha": "50f5173d932e8e61f858120bcb800b97af589f46", "weights": "model.safetensors", "bytes": 374998696},
    "pythia-160m-seed1": {"sha": "5ed55222b3092e69edcfa5fa646b7aac0b7e03d9", "weights": "model.safetensors", "bytes": 374998692},
    "pythia-160m-seed2": {"sha": "6e9b9f6270d1d624f9d7188c1e8909600c4efae2", "weights": "model.safetensors", "bytes": 374998696},
    "pythia-160m-seed3": {"sha": "a74886a25b5f4ffb3b9a0fa80bd02750f13461bd", "weights": "model.safetensors", "bytes": 374998692},
    "pythia-160m-seed4": {"sha": "c7a7ebcc54c75407bfd8cdea09c5cfa2c7961b4d", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed5": {"sha": "7682ee9da4d8d399d3ecc3ad468500b3e99cddf3", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed6": {"sha": "78e11a97c628c5716c07a277de902f76b99695a7", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed7": {"sha": "11b513c3f630f59d0d61cfc0163fc05426f30c07", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed8": {"sha": "2149b1efc141db551191f87f778d1f6979ded3e9", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed9": {"sha": "08020a8fabd0f0b44d45db35910b9c312b9ade3a", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-410m": {"sha": "9879c9b5f8bea9051dcb0e68dff21493d67e9d4f", "weights": "model.safetensors", "bytes": 911373632},
    "pythia-410m-seed1": {"sha": "33803c4f5a1e9a4bece59e52f17bb8755add33e1", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed2": {"sha": "da2549c8d3e03e6e927594b9060073abdb94e6ba", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed3": {"sha": "1dd84c3d638fd5b6e3c77815ca2a26f1cdf1a307", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed4": {"sha": "57c8ecb67375772335b779545246e2f3a4aed69b", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed5": {"sha": "8f2c179eee40284065226c82b9fa0d8106658a6f", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed6": {"sha": "6a2463662069b7a4f5e77cf3c222dc60ff6cd3e4", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed7": {"sha": "84f51f7862c384d3d3d4095fb3b4d0438022b39e", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed8": {"sha": "368c5c53fda3ea210d154fcf6673328fa05da6ca", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed9": {"sha": "509afc6f4768f37ebab1705b93a098ecb5d78855", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-160m-data-seed1": {"sha": "dfa32dc674e75495f4046a721de7025d6610109c", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-data-seed2": {"sha": "3b725b0a0f47ca8acdb9bd4d496ac13a52ccab7a", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-data-seed3": {"sha": "c8baaae0f5e9c59fe8e2c54c9e5b4ad4534871ab", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-weight-seed1": {"sha": "fe3e1d29ca2b321c686a0b43d22bf6ed1c3ba6a2", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-weight-seed2": {"sha": "e93e68ff086ab6a5e8b86c129d55e7b5e9c66ea0", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-weight-seed3": {"sha": "0e6423cce79da02daa7975ceb71d6d8d1d77a305", "weights": "pytorch_model.bin", "bytes": 375036845},
}

'''


In [ ]:
#@title 2.3 Предложения: три шаблона и минимальные пары
PARTS[2] = r'''# ------------------------------------------------------------------ the sentences
HEAD = (("nurse", "nurses"), ("doctor", "doctors"), ("author", "authors"), ("pilot", "pilots"),
        ("artist", "artists"), ("farmer", "farmers"), ("teacher", "teachers"), ("driver", "drivers"))
PEOPLE = (("critic", "critics"), ("guard", "guards"), ("student", "students"),
          ("manager", "managers"), ("friend", "friends"))
NOUN = (("window", "windows"), ("table", "tables"), ("door", "doors"), ("garden", "gardens"),
        ("office", "offices"))
VERB = (("likes", "like"), ("knows", "know"), ("helps", "help"), ("meets", "meet"), ("trusts", "trust"))
MODS = ("new", "old", "small", "quiet")
PREPS = ("near", "behind", "beside")
TEMPLATES = ("rc_obj", "rc_subj", "pp")
LENGTH = {"rc_obj": 7, "rc_subj": 7, "pp": 6}
SUBJ_POS = 2
CARRIERS = {"SUBJ": {"rc_obj": (2,), "rc_subj": (2, 4), "pp": (2,)},
            "ATTR": {"rc_obj": (5, 6), "rc_subj": (6,), "pp": (5,)}}
# columns of a sentence record
TPL, MOD, HEADC, SUBJ_PL, ATT, ATTR_PL, VERBC, PREP = range(8)
NUMBER_COL = {"SUBJ": SUBJ_PL, "ATTR": ATTR_PL}


def all_words() -> list[str]:
    out = ["The", "that", "the", "is", "are"] + list(MODS) + list(PREPS)
    for pairs in (HEAD, PEOPLE, NOUN, VERB):
        for a, b in pairs:
            out += [a, b]
    return out


def vocab_ids(tokenizer) -> dict[str, int]:
    """Every word one token; 'The' opens the sentence without a leading space."""
    ids = {}
    for w in all_words():
        enc = tokenizer.encode(w if w == "The" else " " + w)
        if len(enc) != 1:
            raise SystemExit(f"'{w}' is {len(enc)} tokens for this tokenizer. Stopping.")
        ids[w] = enc[0]
    return ids


def sample(n: int, rng: np.random.Generator, tpl: np.ndarray | None = None) -> np.ndarray:
    """n sentence records; numbers of subject and attractor independent and uniform."""
    s = np.zeros((n, 8), dtype=np.int64)
    s[:, TPL] = rng.integers(0, 3, n) if tpl is None else tpl
    s[:, MOD] = rng.integers(0, len(MODS), n)
    s[:, HEADC] = rng.integers(0, len(HEAD), n)
    s[:, SUBJ_PL] = rng.integers(0, 2, n)
    s[:, ATT] = rng.integers(0, 5, n)              # PEOPLE and NOUN both have 5 lemmas
    s[:, ATTR_PL] = rng.integers(0, 2, n)
    s[:, VERBC] = rng.integers(0, len(VERB), n)
    s[:, PREP] = rng.integers(0, len(PREPS), n)
    return s


def words(rec) -> list[str]:
    t = TEMPLATES[rec[TPL]]
    head = HEAD[rec[HEADC]][rec[SUBJ_PL]]
    mod = MODS[rec[MOD]]
    if t == "rc_obj":
        return ["The", mod, head, "that", "the", PEOPLE[rec[ATT]][rec[ATTR_PL]], VERB[rec[VERBC]][rec[ATTR_PL]]]
    if t == "rc_subj":
        return ["The", mod, head, "that", VERB[rec[VERBC]][rec[SUBJ_PL]], "the", PEOPLE[rec[ATT]][rec[ATTR_PL]]]
    return ["The", mod, head, PREPS[rec[PREP]], "the", NOUN[rec[ATT]][rec[ATTR_PL]]]


def token_ids(recs: np.ndarray, vocab: dict[str, int]) -> torch.Tensor:
    """Records of one template -> (n, T) token ids."""
    return torch.tensor([[vocab[w] for w in words(r)] for r in recs], dtype=torch.long)


def tangent_words(rec, variable: str) -> list[tuple[int, str, str]]:
    """(position, singular-side word, plural-side word) at the variable's carriers."""
    t = TEMPLATES[rec[TPL]]
    out = []
    for pos in CARRIERS[variable][t]:
        if pos == 2:
            sg, pl = HEAD[rec[HEADC]]
        elif t == "pp":
            sg, pl = NOUN[rec[ATT]]
        elif (t, pos) in (("rc_obj", 5), ("rc_subj", 6)):
            sg, pl = PEOPLE[rec[ATT]]
        else:                                        # the clause's verb: singular agreement first
            sg, pl = VERB[rec[VERBC]]
        out.append((pos, sg, pl))
    return out


def make_pairs(variable: str, split: int, n: int):
    """(base, source, counterfactual) records of one split; the same for every model."""
    rng = np.random.default_rng([PAIR_STREAM, VAR_ID[variable], split])
    tpl = rng.integers(0, 3, n)
    base = sample(n, rng, tpl)
    src = sample(n, rng, tpl)
    cf = base.copy()
    col = NUMBER_COL[variable]
    cf[:, col] = src[:, col]
    return base, src, cf


def differs_only_at_carriers(base: np.ndarray, cf: np.ndarray, variable: str, vocab) -> bool:
    for b, c in zip(base, cf):
        wb, wc = words(b), words(c)
        diff = {i for i, (x, y) in enumerate(zip(wb, wc)) if x != y}
        if not diff <= set(CARRIERS[variable][TEMPLATES[b[TPL]]]):
            return False
    return True


'''


In [ ]:
#@title 2.4 Прямой проход GPT-NeoX с кэшем префикса
PARTS[3] = r'''# ------------------------------------------------------------------ the model
def rope_params(cfg) -> tuple[float, float]:
    """(base, rotary share) of the rotary embedding: config.rope_parameters in transformers 5,
    rotary_emb_base / rotary_pct in transformers 4."""
    rp = getattr(cfg, "rope_parameters", None)
    if isinstance(rp, dict) and "rope_theta" in rp:
        if rp.get("rope_type", "default") != "default":
            raise SystemExit(f"rope type {rp.get('rope_type')} is not the one this forward pass implements")
        return float(rp["rope_theta"]), float(rp.get("partial_rotary_factor", 1.0))
    if getattr(cfg, "rope_scaling", None):
        raise SystemExit("rope scaling is not implemented in this forward pass")
    pct = getattr(cfg, "rotary_pct", None)
    pct = getattr(cfg, "partial_rotary_factor", 1.0) if pct is None else pct
    base = getattr(cfg, "rotary_emb_base", None)
    base = getattr(cfg, "rope_theta", 10000) if base is None else base
    return float(base), float(pct)


class Neox:
    """GPT-NeoX forward pass written out, so that a residual can be cached and continued.

    Uses the transformers module weights; control (1) checks it against the transformers
    forward pass on every model.
    """

    def __init__(self, hf, are_id: int, is_id: int):
        self.hf = hf
        self.m = hf.gpt_neox
        self.blocks = list(self.m.layers)
        self.L = len(self.blocks)
        cfg = hf.config
        if not getattr(cfg, "use_parallel_residual", True):
            raise SystemExit("this forward pass assumes the parallel residual of Pythia")
        self.H = cfg.num_attention_heads
        self.hs = cfg.hidden_size // self.H
        self.d = cfg.hidden_size
        base, pct = rope_params(cfg)
        self.rot = int(self.hs * pct)
        dev = next(hf.parameters()).device
        self.inv_freq = 1.0 / (base ** (torch.arange(0, self.rot, 2, dtype=torch.float32, device=dev) / self.rot))
        self.are_id, self.is_id = are_id, is_id
        head = hf.get_output_embeddings()           # embed_out in transformers 4, lm_head in 5
        self.w = (head.weight[are_id] - head.weight[is_id]).detach()
        self.bias = 0.0
        if getattr(head, "bias", None) is not None:
            self.bias = float(head.bias[are_id] - head.bias[is_id])
        self.device = dev

    def rope(self, pos: torch.Tensor):
        freqs = pos[:, None].float() * self.inv_freq[None]
        emb = torch.cat([freqs, freqs], -1)
        return emb.cos(), emb.sin()

    @staticmethod
    def _rotate(x, cos, sin, rot):
        xr, xp = x[..., :rot], x[..., rot:]
        half = rot // 2
        rh = torch.cat([-xr[..., half:], xr[..., :half]], -1)
        return torch.cat([xr * cos + rh * sin, xp], -1)

    def block(self, j: int, h: torch.Tensor, pos: torch.Tensor, past=None):
        """One block on the positions `pos` of h (n, t, d); past = (K, V) of the earlier positions."""
        blk = self.blocks[j]
        n, t, _ = h.shape
        x = blk.input_layernorm(h)
        qkv = blk.attention.query_key_value(x).view(n, t, self.H, 3 * self.hs).transpose(1, 2)
        q, k, v = qkv.chunk(3, -1)
        cos, sin = self.rope(pos)
        q = self._rotate(q, cos.to(q.dtype), sin.to(q.dtype), self.rot)
        k = self._rotate(k, cos.to(k.dtype), sin.to(k.dtype), self.rot)
        if past is not None:
            k = torch.cat([past[0], k], 2)
            v = torch.cat([past[1], v], 2)
        scores = (q @ k.transpose(-1, -2)) * self.hs ** -0.5
        kpos = torch.arange(k.shape[2], device=h.device)
        mask = kpos[None, :] > pos[:, None]
        scores = scores.masked_fill(mask, float("-inf"))
        low = scores.dtype in (torch.float16, torch.bfloat16)        # as transformers: softmax in fp32
        probs = torch.softmax(scores.float() if low else scores, -1).to(q.dtype)
        o = (probs @ v).transpose(1, 2).reshape(n, t, self.d)
        a = blk.attention.dense(o)
        m = blk.mlp(blk.post_attention_layernorm(h))
        return h + a + m, (k, v)

    def readout(self, h_last: torch.Tensor) -> torch.Tensor:
        return self.m.final_layer_norm(h_last) @ self.w + self.bias

    def embed(self, ids: torch.Tensor) -> torch.Tensor:
        return self.m.embed_in(ids.to(self.device))

    def forward(self, ids: torch.Tensor, patch=None) -> torch.Tensor:
        """B(x); patch = (r, p, new residual (n, d)) replaces the residual at index r, position p."""
        h = self.embed(ids)
        T = h.shape[1]
        pos = torch.arange(T, device=self.device)
        for j in range(self.L + 1):
            if patch is not None and patch[0] == j:
                h = h.clone()
                h[:, patch[1]] = patch[2]
            if j == self.L:
                break
            h, _ = self.block(j, h, pos)
        return self.readout(h[:, -1])

    def resid(self, h: torch.Tensor, r: int) -> torch.Tensor:
        """The residual stream at index r of every position, from the embeddings h."""
        pos = torch.arange(h.shape[1], device=h.device)
        for j in range(r):
            h, _ = self.block(j, h, pos)
        return h

    def prefix(self, ids: torch.Tensor, r: int, p: int) -> dict:
        """The cache for continuing from (r, p): the residual at index r for positions p..T-1 and
        the keys and values of blocks r..L-1 at positions 0..p-1."""
        with torch.no_grad():
            h = self.embed(ids)
            T = h.shape[1]
            pos = torch.arange(T, device=self.device)
            for j in range(r):
                h, _ = self.block(j, h, pos)
            hr = h[:, p:].clone()
            past = []
            hh = h
            for j in range(r, self.L):
                hh, (k, v) = self.block(j, hh, pos)
                past.append((k[:, :, :p].clone(), v[:, :, :p].clone()))
        return {"h": hr, "past": past, "r": r, "p": p, "T": T}

    def cont(self, cache: dict, new_p: torch.Tensor, idx=None) -> torch.Tensor:
        """B with the residual at (r, p) replaced by new_p (n, d); idx selects rows of the cache."""
        r, p, T = cache["r"], cache["p"], cache["T"]
        h = cache["h"] if idx is None else cache["h"][idx]
        h = torch.cat([new_p[:, None].to(h.dtype), h[:, 1:]], 1)
        pos = torch.arange(p, T, device=h.device)
        for j in range(r, self.L):
            k, v = cache["past"][j - r]
            if idx is not None:
                k, v = k[idx], v[idx]
            h, _ = self.block(j, h, pos, past=(k, v) if p > 0 else None)
        return self.readout(h[:, -1])


def slice_cache(cache: dict, idx) -> dict:
    return {"h": cache["h"][idx], "past": [(k[idx], v[idx]) for k, v in cache["past"]],
            "r": cache["r"], "p": cache["p"], "T": cache["T"]}


'''


In [ ]:
#@title 2.5 Наборы пар, подмена на последнем токене, DAS
PARTS[4] = r'''# ------------------------------------------------------------------ batched helpers over mixed templates
CHUNK = 512


def by_template(recs: np.ndarray) -> dict[int, np.ndarray]:
    return {t: np.flatnonzero(recs[:, TPL] == t) for t in range(3) if (recs[:, TPL] == t).any()}


def outputs(model: Neox, recs: np.ndarray, vocab) -> torch.Tensor:
    out = torch.empty(len(recs), device=model.device)
    with torch.no_grad():
        for t, ix in by_template(recs).items():
            for s in range(0, len(ix), CHUNK):
                part = ix[s:s + CHUNK]
                out[torch.as_tensor(part, device=model.device)] = model.forward(token_ids(recs[part], vocab)).float()
    return out


def point(t: int, where: str) -> int:
    return LENGTH[TEMPLATES[t]] - 1 if where == "last" else SUBJ_POS


class PairSet:
    """One split of minimal pairs on one model, ready for the interchange at (r, where)."""

    def __init__(self, model: Neox, vocab, base, src, cf, r: int, where: str, sigma: float):
        self.model, self.r, self.where = model, r, where
        self.groups = by_template(base)
        self.n = len(base)
        self.y_b = outputs(model, base, vocab)
        self.y_cf = outputs(model, cf, vocab)
        self.sigma = sigma
        self.cache, self.h_s, self.local = {}, {}, np.zeros(self.n, dtype=np.int64)
        for t, ix in self.groups.items():
            p = point(t, where)
            self.cache[t] = model.prefix(token_ids(base[ix], vocab), r, p)
            with torch.no_grad():
                self.h_s[t] = model.resid(model.embed(token_ids(src[ix], vocab)), r)[:, p]
            self.local[ix] = np.arange(len(ix))
        self.order = np.concatenate([self.groups[t] for t in self.groups])
        dev = model.device
        self.yo_cf = self.y_cf[torch.as_tensor(self.order, device=dev)].double().cpu().numpy()
        self.yo_b = self.y_b[torch.as_tensor(self.order, device=dev)].double().cpu().numpy()

    def predict(self, q: torch.Tensor) -> np.ndarray:
        """B(h') for every pair, in the order self.order, for the unit direction q."""
        preds = []
        with torch.no_grad():
            for t, ix in self.groups.items():
                for s in range(0, len(ix), CHUNK):
                    li = torch.arange(s, min(s + CHUNK, len(ix)), device=self.model.device)
                    hb = self.cache[t]["h"][li, 0]
                    hs = self.h_s[t][li]
                    new = hb + ((hs - hb) @ q)[:, None] * q[None]
                    preds.append(self.model.cont(self.cache[t], new, idx=li))
        return torch.cat(preds).double().cpu().numpy()

    def informative(self) -> int:
        return int((np.abs(self.yo_cf - self.yo_b) >= TOL_INFORMATIVE * self.sigma).sum())

    def r2(self, q: torch.Tensor) -> tuple[float, int, float]:
        """(interchange R^2, informative pairs, decision IIA) over the whole split."""
        pr, y_cf, y_b = self.predict(q), self.yo_cf, self.yo_b
        den = float(((y_cf - y_b) ** 2).sum())
        val = float(1.0 - ((pr - y_cf) ** 2).sum() / den) if den > 0 else float("nan")
        flip = np.sign(y_cf) != np.sign(y_b)
        dec = float((np.sign(pr[flip]) == np.sign(y_cf[flip])).mean()) if flip.any() else float("nan")
        return val, self.informative(), dec

    def r2_by_template(self, q: torch.Tensor) -> dict[str, float]:
        pr = self.predict(q)
        out, start = {}, 0
        for t, ix in self.groups.items():
            sl = slice(start, start + len(ix))
            start += len(ix)
            y_cf, y_b = self.yo_cf[sl], self.yo_b[sl]
            den = float(((y_cf - y_b) ** 2).sum())
            out[TEMPLATES[t]] = float(1.0 - ((pr[sl] - y_cf) ** 2).sum() / den) if den > 0 else float("nan")
        return out


def seed_of(*parts) -> int:
    return int(np.random.SeedSequence([int(p) for p in parts]).generate_state(1)[0])


def das_steps(train: PairSet, c: dict, g: torch.Generator, steps: int) -> torch.Tensor:
    """Adam on the interchange loss from a random start; returns the unit direction."""
    w = torch.randn(train.model.d, generator=g).to(train.model.device).requires_grad_(True)
    opt = torch.optim.Adam([w], lr=c["lr"])
    for _ in range(steps):
        rows = torch.randint(0, train.n, (min(c["batch"], train.n),), generator=g).numpy()
        q = w / w.norm()
        loss = 0.0
        for t, ix in train.groups.items():
            sel = rows[np.isin(rows, ix)]
            if len(sel) == 0:
                continue
            loc = torch.as_tensor(train.local[sel], device=train.model.device)
            hb = train.cache[t]["h"][loc, 0]
            hs = train.h_s[t][loc]
            new = hb + ((hs - hb) @ q)[:, None] * q[None]
            pred = train.model.cont(train.cache[t], new, idx=loc)
            ref = train.y_cf[torch.as_tensor(sel, device=train.model.device)]
            loss = loss + (((pred - ref) / train.sigma) ** 2).sum()
        loss = loss / len(rows)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return (w / w.norm()).detach()


def fit_das(train: PairSet, val: PairSet, c: dict, seed_parts: tuple) -> tuple[torch.Tensor, float]:
    """Stage 2.0's DAS at k = 1; the restart with the best validation R^2 (ties: the lower index)."""
    best = None
    for rs in range(c["restarts"]):
        g = torch.Generator().manual_seed(seed_of(*seed_parts, rs))
        q = das_steps(train, c, g, c["steps"])
        v = val.r2(q)[0]
        key = (-np.inf if np.isnan(v) else v, -rs)
        if best is None or key > best[0]:
            best = (key, q, v)
    return best[1], best[2]


'''


In [ ]:
#@title 2.6 Формулы с нулём пар: без меток и с метками
PARTS[5] = r'''# ------------------------------------------------------------------ formulas with zero pairs
def formula_inputs(model: Neox, recs: np.ndarray, vocab, variable: str, r: int, where: str) -> dict:
    """Residuals H, gradients G = dB/dh and Jacobian-vector products D along the plural direction
    at (r, where), and the variable's labels, on the model's own sentences."""
    H, G, D, y = [], [], [], []
    emb = model.m.embed_in.weight
    for t, ix in by_template(recs).items():
        p = point(t, where)
        for s in range(0, len(ix), CHUNK):
            part = recs[ix[s:s + CHUNK]]
            ids = token_ids(part, vocab).to(model.device)
            cache = model.prefix(ids, r, p)
            hp = cache["h"][:, 0].clone().requires_grad_(True)
            out = model.cont(cache, hp)
            g = torch.autograd.grad(out.sum(), hp)[0]
            e0 = model.embed(ids).detach()
            tan = torch.zeros_like(e0)
            vid = {w: vocab[w] for w in vocab}
            for n_, rec in enumerate(part):
                for pos, sg, pl in tangent_words(rec, variable):
                    tan[n_, pos] = (emb[vid[pl]] - emb[vid[sg]]).detach()
            fn = lambda e: model.resid(e, r)[:, p]
            hval, dval = torch.func.jvp(fn, (e0,), (tan,))
            H.append(hval.detach().double().cpu())
            D.append(dval.detach().double().cpu())
            G.append(g.detach().double().cpu())
            y.append(torch.as_tensor(part[:, NUMBER_COL[variable]], dtype=torch.float64))
    return {"H": torch.cat(H), "G": torch.cat(G), "D": torch.cat(D), "y": torch.cat(y)}


def _unit(v: torch.Tensor) -> torch.Tensor:
    return v / v.norm().clamp(min=1e-30)


def formulas(fi: dict) -> dict[str, torch.Tensor]:
    H, G, D, y = fi["H"], fi["G"], fi["D"], fi["y"]
    d = H.shape[1]
    Hc = H - H.mean(0)
    cov = Hc.T @ Hc / len(H)
    out = {"pca": _unit(torch.linalg.eigh(cov)[1][:, -1]),
           "first_order": _unit(D.mean(0)),
           "gradient": _unit(G.mean(0))}
    m1, m0 = H[y == 1].mean(0), H[y == 0].mean(0)
    out["diffmean"] = _unit(m1 - m0)
    X1, X0 = H[y == 1] - m1, H[y == 0] - m0
    sw = (X1.T @ X1 + X0.T @ X0) / len(H)
    lam = 1e-3 * torch.trace(sw) / d
    out["lda"] = _unit(torch.linalg.solve(sw + lam * torch.eye(d, dtype=sw.dtype), m1 - m0))
    scale = Hc.pow(2).mean().sqrt().clamp(min=1e-30)
    X = Hc / scale
    w = torch.zeros(d, dtype=X.dtype, requires_grad=True)
    b = torch.zeros(1, dtype=X.dtype, requires_grad=True)
    opt = torch.optim.LBFGS([w, b], max_iter=500, tolerance_grad=1e-10, tolerance_change=1e-12,
                            line_search_fn="strong_wolfe")
    sgn = 2 * y - 1

    def closure():
        opt.zero_grad()
        loss = F.softplus(-sgn * (X @ w + b)).mean() + 0.5e-3 * (w * w).sum()
        loss.backward()
        return loss
    opt.step(closure)
    out["probe"] = _unit(w.detach())
    return out


def random_dirs(d: int, size: str, seed: int, variable: str) -> list[torch.Tensor]:
    rng = np.random.default_rng([RANDOM_STREAM, SIZE_ID[size], seed, VAR_ID[variable]])
    return [_unit(torch.as_tensor(rng.standard_normal(d))) for _ in range(N_RANDOM)]


'''


In [ ]:
#@title 2.7 Статистика: перестановочный тест, Холм, бутстрэп
PARTS[6] = r'''# ------------------------------------------------------------------ statistics (as mint.eval.stats)
def paired_sign_flip(a, b) -> tuple[float, float]:
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    d = d[np.isfinite(d)]
    m = len(d)
    if m == 0:
        return float("nan"), float("nan")
    obs = abs(d.mean())
    signs = np.array(list(product([1, -1], repeat=m)), dtype=float)
    null = np.abs((signs * d).mean(axis=1))
    return float(d.mean()), float((null >= obs - 1e-12).mean())


def holm(pvals: list[float], alpha: float = 0.05) -> list[dict]:
    idx = [i for i, p in enumerate(pvals) if np.isfinite(p)]
    order = sorted(idx, key=lambda i: pvals[i])
    n = len(order)
    out = [{"p": p, "p_adj": float("nan"), "reject": False} for p in pvals]
    running = 0.0
    for rank, i in enumerate(order):
        adj = min(1.0, max(running, (n - rank) * pvals[i]))
        running = adj
        out[i] = {"p": pvals[i], "p_adj": adj, "reject": bool(adj <= alpha)}
    return out


def boot(vals, n: int = N_BOOT) -> list[float]:
    v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
    if len(v) == 0:
        return [float("nan"), float("nan")]
    rng = np.random.default_rng(0)
    means = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]


'''


In [ ]:
#@title 2.8 Среда, детерминизм, загрузка моделей
PARTS[7] = r'''# ------------------------------------------------------------------ environment and models
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def set_determinism():
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def environment(precision: dict) -> dict:
    import transformers
    cuda = torch.cuda.is_available()
    return {"script_sha256": sha256_file(__file__), "torch": torch.__version__,
            "transformers": transformers.__version__, "numpy": np.__version__,
            "python": platform.python_version(), "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if cuda else None,
            "colab": "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG")),
            "precision": precision,
            "determinism": {"use_deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
                            "warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
                            "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
                            "cudnn_deterministic": torch.backends.cudnn.deterministic,
                            "cudnn_benchmark": torch.backends.cudnn.benchmark,
                            "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
                            "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32}}


def repo(name: str) -> str:
    return f"EleutherAI/{name}"


def check_name(size: str, seed: int) -> str:
    return f"pythia-{size}-seed{seed}"


def load(name: str, device, fp16_autocast: bool = False):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    rev = REVISIONS.get(name, {}).get("sha")
    if rev is None:
        raise SystemExit(f"{name}: no pinned revision. Stopping.")
    tk = AutoTokenizer.from_pretrained(repo(name), revision=rev)
    hf = AutoModelForCausalLM.from_pretrained(repo(name), revision=rev)
    hf = hf.float().to(device).eval()          # fp32 whatever the checkpoint and the library default
    for p in hf.parameters():
        p.requires_grad_(False)
    vocab = vocab_ids(tk)
    return hf, Neox(hf, vocab["are"], vocab["is"]), vocab, rev


'''


In [ ]:
#@title 2.9 Контроли (в float64)
PARTS[8] = r'''# ------------------------------------------------------------------ controls
def controls(hf, model: Neox, vocab, r: int, rng_seed: int = 0) -> dict:
    """The protocol's controls (1)-(3), computed in float64, where they are exact up to rounding;
    and, descriptive, how far fp32 rounding moves B on the same sentences. The weights go
    fp32 -> fp64 -> fp32, which is exact."""
    rng = np.random.default_rng([PROBE_STREAM, 99, rng_seed])
    worst = {"forward": 0.0, "continue": 0.0, "swap": 0.0}
    sets = []
    hf.double()
    try:
        m64 = Neox(hf, model.are_id, model.is_id)
        with torch.no_grad():
            for t in range(3):
                ids = token_ids(sample(64, rng, np.full(64, t)), vocab).to(model.device)
                ids_s = token_ids(sample(64, rng, np.full(64, t)), vocab).to(model.device)
                mine = m64.forward(ids)
                logits = hf(input_ids=ids).logits[:, -1]
                ref = logits[:, vocab["are"]] - logits[:, vocab["is"]]
                worst["forward"] = max(worst["forward"], float(((mine - ref).abs() / (1 + ref.abs())).max()))
                for where in ("last", "subj"):
                    p = point(t, where)
                    cache = m64.prefix(ids, r, p)
                    again = m64.cont(cache, cache["h"][:, 0])
                    worst["continue"] = max(worst["continue"], float(((again - mine).abs() / (1 + mine.abs())).max()))
                    hs = m64.resid(m64.embed(ids_s), r)[:, p]
                    via_cache = m64.cont(cache, hs)
                    via_patch = m64.forward(ids, patch=(r, p, hs))
                    worst["swap"] = max(worst["swap"], float(((via_cache - via_patch).abs() / (1 + via_patch.abs())).max()))
                sets.append((ids, mine))
    finally:
        hf.float()
    with torch.no_grad():
        rounding = max(float(((model.forward(ids).double() - mine).abs() / (1 + mine.abs())).max()) for ids, mine in sets)
    return {"float64": worst, "fp32_rounding": rounding}


def controls_failed(ctl: dict) -> bool:
    return max(ctl["float64"].values()) > CONTROL_TOL


'''


In [ ]:
#@title 2.10 Расчёт одной модели
PARTS[9] = r'''# ------------------------------------------------------------------ one model
def probe_sigma(model: Neox, vocab) -> float:
    recs = sample(N_PROBE, np.random.default_rng([PROBE_STREAM]))
    return float(outputs(model, recs, vocab).std().item())


def pair_sets(model, vocab, variable, r, where, sigma, c, splits=(0, 1, 2)):
    sizes = (c["n_train"], c["n_val"], c["n_test"])
    out = {}
    for split in splits:
        if sizes[split] == 0:
            continue
        base, src, cf = make_pairs(variable, split, sizes[split])
        if not differs_only_at_carriers(base, cf, variable, vocab):
            raise SystemExit("control (4): a minimal pair differs outside the carriers. Stopping.")
        out[split] = PairSet(model, vocab, base, src, cf, r, where, sigma)
    return out


def evaluate_point(model, vocab, size, seed, variable, r, where, sigma, c, test_split=2,
                   with_das=True) -> dict:
    sets = pair_sets(model, vocab, variable, r, where, sigma, c)
    ev = sets[test_split]
    rec = {"r": r, "where": where}
    if with_das:
        q, val = fit_das(sets[0], sets[1], c, (TORCH_STREAM, SIZE_ID[size], seed, VAR_ID[variable], r,
                                               0 if where == "last" else 1))
        rec["das"], rec["n_informative"], rec["das_decision_iia"] = ev.r2(q)
        rec["das_validation"] = val
        rec["das_by_template"] = ev.r2_by_template(q)
    fi = formula_inputs(model, sample(c.get("n_formula", N_FORMULA), np.random.default_rng([FORMULA_STREAM])),
                        vocab, variable, r, where)
    for name, v in formulas(fi).items():
        rec[name] = ev.r2(v.to(model.device, torch.float32))[0]
    rec["random"] = float(np.nanmean([ev.r2(v.to(model.device, torch.float32))[0]
                                      for v in random_dirs(model.d, size, seed, variable)]))
    rec["n_informative"] = ev.informative()
    return rec


def run_model(name: str, size: str, seed: int, r: int, c: dict, device, out_dir: str,
              precision: str = "fp32") -> dict:
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{name}.json")
    if os.path.exists(path):
        old = json.load(open(path))
        sha = old["environment"]["script_sha256"]
        if sha == sha256_file(__file__) or (size in SIZES and sha == PROTOCOL_SHA256):
            print(f"{name}: already done, kept", flush=True)
            return old
    t0 = time.time()
    hf, model, vocab, rev = load(name, device)
    ctl = controls(hf, model, vocab, r)
    if controls_failed(ctl):
        raise SystemExit(f"{name}: control failed {ctl}. Stopping.")
    sigma = probe_sigma(model, vocab)
    rec = {"model": name, "repo": repo(name), "revision": rev, "size": size, "seed": seed,
           "checkpoint_dtype": str(getattr(hf.config, "dtype", None) or getattr(hf.config, "torch_dtype", None)),
           "sigma": sigma, "controls": ctl,
           "point": {"last": r, "arrival": R_ARRIVAL.get(size, r), "diagnostic_subject_position": R_DIAG}}
    ac = torch.autocast("cuda", dtype=torch.float16) if precision == "fp16" else torch.autocast("cpu", enabled=False)
    with ac:
        for var in VARIABLES:
            rec[var] = evaluate_point(model, vocab, size, seed, var, r, "last", sigma, c)
            print(f"  {name} {var}: DAS {rec[var]['das']:+.3f} " +
                  " ".join(f"{k} {rec[var][k]:+.3f}" for k in FORMULAS + ('random',)), flush=True)
        ra = R_ARRIVAL.get(size, r)
        if ra != r:
            rec["SUBJ_arrival"] = evaluate_point(model, vocab, size, seed, "SUBJ", ra, "last", sigma, c)
            print(f"  {name} SUBJ at arrival r={ra}: DAS {rec['SUBJ_arrival']['das']:+.3f}", flush=True)
        else:
            rec["SUBJ_arrival"] = dict(rec["SUBJ"], same_as_last=True)     # 14m: the two points coincide
        rec["SUBJ_subject_position"] = evaluate_point(model, vocab, size, seed, "SUBJ", R_DIAG, "subj", sigma, c)
    rec["seconds"] = time.time() - t0
    rec["environment"] = environment({"mode": precision, "autocast_fp16": precision == "fp16"})
    with open(path + ".tmp", "w") as f:
        json.dump(rec, f, indent=1)
    os.replace(path + ".tmp", path)
    del hf, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rec


'''


In [ ]:
#@title 2.11 Пилот, замер, основной прогон, сводка
PARTS[10] = r'''# ------------------------------------------------------------------ pilot, time, smoke, run, verify
def pilot(device, c: dict, out: str, models=PILOT_MODELS, smoke: bool = False,
          parts: str = "pilot_parts") -> dict:
    """The layer sweep on the pilot models; one file per model in `parts`, so that a broken
    session resumes where it stopped (a file is reused only if the script and config match)."""
    res = {}
    os.makedirs(parts, exist_ok=True)
    me = sha256_file(__file__)
    for name in models:
        size = name.split("-")[1]
        path = os.path.join(parts, f"{name}.json")
        if os.path.exists(path):
            old = json.load(open(path))
            if old.get("script_sha256") == me and old.get("config") == c:
                old["profile"] = {int(k): v for k, v in old["profile"].items()}
                res[name] = old
                print(f"  pilot {name}: already done, kept", flush=True)
                continue
        hf, model, vocab, rev = load(name, device)
        sigma = probe_sigma(model, vocab)
        rs = list(range(model.L + 1)) if size != SIZE_410 else list(range(0, model.L + 1, 2))
        if smoke:
            rs = rs[:2]
        prof = {}
        for r in rs:
            ctl = controls(hf, model, vocab, r)
            if controls_failed(ctl):
                raise SystemExit(f"{name} r={r}: control failed {ctl}. Stopping.")
            sets = pair_sets(model, vocab, "SUBJ", r, "last", sigma, c, splits=(0, 1))
            q, val = fit_das(sets[0], sets[1], c, (TORCH_STREAM, 90 + SIZE_ID[size], len(res), 0, r, 0))
            fi = formula_inputs(model, sample(c.get("n_formula", N_FORMULA),
                                              np.random.default_rng([FORMULA_STREAM])), vocab, "SUBJ", r, "last")
            row = {"das": val, "controls": ctl}
            for k, v in formulas(fi).items():
                row[k] = sets[1].r2(v.to(model.device, torch.float32))[0]
            prof[r] = row
            print(f"  pilot {name} r={r}: DAS {val:+.3f} " +
                  " ".join(f"{k} {row[k]:+.3f}" for k in FORMULAS), flush=True)
        res[name] = {"size": size, "revision": rev, "sigma": sigma, "profile": prof,
                     "script_sha256": me, "config": c, "environment": environment({"mode": "fp32"})}
        with open(path + ".tmp", "w") as f:
            json.dump(res[name], f, indent=1)
        os.replace(path + ".tmp", path)
        del hf, model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    # the rule written before the pilot
    choice = {}
    for size in SIZES + (SIZE_410,):
        names = [n for n in res if res[n]["size"] == size]
        if not names:
            continue
        rs = sorted(res[names[0]]["profile"])
        mean = {r: float(np.nanmean([res[n]["profile"][r]["das"] for n in names])) for r in rs}
        best = max(rs, key=lambda r: (mean[r], -r))
        choice[size] = {"r_star": best, "mean_validation_das": mean,
                        "formulas_at_r_star": {k: float(np.nanmean([res[n]["profile"][best][k] for n in names]))
                                               for k in FORMULAS}}
    pooled = {k: float(np.nanmean([choice[s]["formulas_at_r_star"][k] for s in SIZES if s in choice]))
              for k in FORMULAS}
    strongest = {"unlabelled": max(UNLABELLED, key=lambda k: pooled[k]),
                 "labelled": max(LABELLED, key=lambda k: pooled[k])}
    doc = {"pilot": res, "choice": choice, "formulas_pooled_14m_160m": pooled, "strongest": strongest,
           "config": c, "environment": environment({"mode": "fp32"})}
    if out:
        with open(out, "w") as f:
            json.dump(doc, f, indent=1)
    print("r* per size:", {s: v["r_star"] for s, v in choice.items()}, "strongest:", strongest, flush=True)
    return doc


def time_410(device) -> dict:
    """A short measurement of one DAS step at batch 256 for 160m and 410m, fp32 and fp16 autocast."""
    out = {}
    for name in ("pythia-160m", "pythia-410m"):
        hf, model, vocab, rev = load(name, device)
        r = model.L // 2
        sets = pair_sets(model, vocab, "SUBJ", r, "last", 1.0, dict(FULL, n_train=4096, n_val=0, n_test=0),
                         splits=(0,))
        for mode in ("fp32", "fp16"):
            ac = torch.autocast("cuda", dtype=torch.float16) if mode == "fp16" else torch.autocast("cpu", enabled=False)
            with ac:
                das_steps(sets[0], FULL, torch.Generator().manual_seed(0), 3)      # warm-up
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                t0 = time.time()
                das_steps(sets[0], FULL, torch.Generator().manual_seed(1), 20)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                step = (time.time() - t0) / 20
            # the run per model: 2 variables + the diagnostic, 3 restarts x 500 steps each
            per_model_h = 3 * 3 * 500 * step / 3600
            out[f"{name}|{mode}"] = {"seconds_per_step": step, "das_hours_per_model": per_model_h,
                                     "das_hours_9_seeds": 9 * per_model_h}
            print(f"  {name} {mode}: {step:.3f} s per step; DAS for 9 seeds about {9 * per_model_h:.1f} h", flush=True)
        if torch.cuda.is_available():
            out[f"{name}|peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 1e9
        del hf, model
    out["environment"] = environment({"mode": "fp32 and fp16 autocast"})
    with open("pythia_address_time.json", "w") as f:
        json.dump(out, f, indent=1)
    return out


def units(records: list[dict], variable: str, arm: str, key: str = None) -> list[float]:
    out = []
    for seed in CHECK_SEEDS:
        vals = []
        for r in records:
            if r["seed"] != seed or r["size"] not in SIZES:
                continue
            rec = r[key or variable]
            if rec["n_informative"] >= MIN_INFORMATIVE:
                vals.append(rec[arm])
        out.append(float(np.nanmean(vals)) if vals else float("nan"))
    return out


def verdict(row: dict) -> str:
    lo, hi = row["ci95_difference"]
    if lo >= -MARGIN and hi <= MARGIN:
        return "ANALYTIC"
    if row["reject"] and row["delta"] > 0:
        return "DAS HIGHER"
    if row["reject"] and row["delta"] < 0:
        return "FORMULA HIGHER"
    return "UNDETERMINED"


def aggregate(records: list[dict]) -> dict:
    arms = ("das", "random") + FORMULAS
    u = {f"SUBJ@{pt}": {arm: units(records, "SUBJ", arm, key=key) for arm in arms} for pt, key in POINTS.items()}
    u["ATTR@last"] = {arm: units(records, "ATTR", arm) for arm in arms}
    # (a) existence: three comparisons, Holm over the three
    rows_a, ps = [], []
    for name in ("SUBJ@last", "SUBJ@arrival", "ATTR@last"):
        delta, p = paired_sign_flip(u[name]["das"], u[name]["random"])
        rows_a.append({"variable_point": name, "reference": "random", "delta": delta, "p": p})
        ps.append(p)
    for r, h in zip(rows_a, holm(ps)):
        r.update(p_adj=h["p_adj"], reject=bool(h["reject"] and r["delta"] > 0))
    read_a = {r["variable_point"]: ("CONFIRMED" if r["reject"] and np.nanmean(u[r["variable_point"]]["das"]) >= BAR
                                    else "NOT CONFIRMED") for r in rows_a}
    # (b) analytic: per point, DAS vs the six formulas, Holm over the six of that point
    rows_b, read_b = {}, {}
    for pt in POINTS:
        rows, ps = [], []
        for f in FORMULAS:
            a, b = u[f"SUBJ@{pt}"]["das"], u[f"SUBJ@{pt}"][f]
            delta, p = paired_sign_flip(a, b)
            diff = [x - y for x, y in zip(a, b) if np.isfinite(x) and np.isfinite(y)]
            rows.append({"formula": f, "uses_labels": f in LABELLED, "delta": delta, "p": p,
                         "ci95_difference": boot(diff)})
            ps.append(p)
        for r, h in zip(rows, holm(ps)):
            r.update(p_adj=h["p_adj"], reject=bool(h["reject"]))
            r["verdict"] = verdict(r)
        vs = [r["verdict"] for r in rows]
        read_b[pt] = ("ADDRESS ANALYTIC" if any(v in ("ANALYTIC", "FORMULA HIGHER") for v in vs)
                      else "DAS HIGHER THAN EVERY FORMULA" if all(v == "DAS HIGHER" for v in vs) else "MIXED")
        rows_b[pt] = rows
    r1_points = [pt for pt in POINTS if read_a[f"SUBJ@{pt}"] == "CONFIRMED"]
    means = {k: {arm: {"mean": float(np.nanmean(x)), "ci95": boot(x)} for arm, x in v.items()} for k, v in u.items()}
    per_size = {}
    for s in sorted({r["size"] for r in records}):
        per_size[s] = {}
        for name, key, var in (("SUBJ@last", "SUBJ", "SUBJ"), ("SUBJ@arrival", "SUBJ_arrival", "SUBJ"),
                               ("ATTR@last", "ATTR", "ATTR")):
            sel = [r[key] for r in records if r["size"] == s and r[key]["n_informative"] >= MIN_INFORMATIVE]
            per_size[s][name] = {arm: float(np.nanmean([x[arm] for x in sel])) if sel else float("nan") for arm in arms}
    diag = {arm: units(records, "SUBJ", arm, key="SUBJ_subject_position") for arm in ("das",) + FORMULAS}
    return {"unit_values": u, "means": means, "existence": rows_a, "read_out_existence": read_a,
            "analytic": rows_b, "read_out_analytic": read_b, "R1_points": r1_points,
            "stop": not r1_points, "per_size": per_size, "subject_position_units": diag,
            "subject_position_means": {arm: float(np.nanmean(x)) for arm, x in diag.items()}}


def pilot_summary(doc: dict) -> dict:
    """From the pilot file: per size the DAS profile at the last token (mean over that size's pilot
    models), the last layer r* (rule written before the pilot), the arrival layer (first r whose mean
    DAS >= ARRIVAL_BAR, criterion chosen after the pilot), the formulas at both points and the
    strongest formula without and with labels at each point (mean over the size, then over
    14m-160m)."""
    out = {"sizes": {}}
    for size in SIZES + (SIZE_410,):
        names = [n for n, m in doc["pilot"].items() if m["size"] == size]
        if not names:
            continue
        rs = sorted(int(r) for r in doc["pilot"][names[0]]["profile"])
        prof = {r: float(np.nanmean([doc["pilot"][n]["profile"][str(r)]["das"] for n in names])) for r in rs}
        last = max(rs, key=lambda r: (prof[r], -r))
        arrival = min(r for r in rs if prof[r] >= ARRIVAL_BAR)
        at = {pt: {k: float(np.nanmean([doc["pilot"][n]["profile"][str(r)][k] for n in names]))
                   for k in ("das",) + FORMULAS} for pt, r in (("last", last), ("arrival", arrival))}
        out["sizes"][size] = {"models": names, "profile": prof, "last": last, "arrival": arrival, "at": at}
    out["pooled"] = {pt: {k: float(np.nanmean([out["sizes"][s]["at"][pt][k] for s in SIZES]))
                          for k in ("das",) + FORMULAS} for pt in POINTS}
    out["strongest"] = {pt: {"unlabelled": max(UNLABELLED, key=lambda k: out["pooled"][pt][k]),
                             "labelled": max(LABELLED, key=lambda k: out["pooled"][pt][k])} for pt in POINTS}
    return out


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--time", action="store_true")
    ap.add_argument("--pilot", action="store_true")
    ap.add_argument("--out", default="R0")
    ap.add_argument("--verify", default=None)
    ap.add_argument("--pilot-summary", default=None, help="print the choices derived from a pilot file")
    a = ap.parse_args(argv)
    if a.verify:
        return verify(a.verify)
    if a.pilot_summary:
        ps = pilot_summary(json.load(open(a.pilot_summary)))
        for size, v in ps["sizes"].items():
            print(f"{size:5s} last r={v['last']:2d} arrival r={v['arrival']:2d}  DAS by r: " +
                  " ".join(f"{r}:{x:.2f}" for r, x in v["profile"].items()))
        print("pooled 14m-160m:", {pt: {k: round(x, 3) for k, x in d.items()} for pt, d in ps["pooled"].items()})
        print("strongest:", ps["strongest"])
        same = ({s: v["last"] for s, v in ps["sizes"].items()} == R_STAR and
                {s: v["arrival"] for s, v in ps["sizes"].items()} == R_ARRIVAL and ps["strongest"] == STRONGEST)
        print("the protocol's constants agree with this file:", same)
        return 0 if same else 5
    set_determinism()
    torch.set_num_threads(4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    started = datetime.now(timezone.utc).isoformat(timespec="seconds")
    t0 = time.time()
    if a.smoke:
        pilot(device, SMOKE, "", models=("pythia-14m",), smoke=True, parts="smoke_tmp/pilot_parts")
        run_model("pythia-14m-seed1", "14m", 1, 1, SMOKE, device, "smoke_tmp")
        print(f"smoke done ({time.time() - t0:.0f}s), nothing kept but smoke_tmp/")
        return 0
    if a.time:
        time_410(device)
        return 0
    if a.pilot:
        pilot(device, PILOT, "pythia_address_pilot.json")
        print(f"pilot done ({time.time() - t0:.0f}s)")
        return 0
    if not R_STAR or not R_ARRIVAL or WITH_410M is None or STRONGEST["last"]["unlabelled"] is None:
        print("the pilot's choices are not in the protocol yet. Stopping.")
        return 3
    os.makedirs(a.out, exist_ok=True)

    def provenance(records):
        envs = {json.dumps({k: v for k, v in r["environment"].items() if k != "precision"}, sort_keys=True)
                for r in records}
        return {"started_utc": started, "environments": [json.loads(e) for e in envs],
                "script_sha256": sha256_file(__file__), "protocol_commit": PROTOCOL_COMMIT,
                "protocol_script_sha256": PROTOCOL_SHA256, "revisions": REVISIONS,
                "r_last": R_STAR, "r_arrival": R_ARRIVAL, "r_diag": R_DIAG,
                "strongest_for_R1": STRONGEST, "with_410m": WITH_410M, "config": FULL,
                "streams": {"pairs": PAIR_STREAM, "formula": FORMULA_STREAM, "probe": PROBE_STREAM,
                            "random": RANDOM_STREAM, "torch": TORCH_STREAM},
                "unit": "seed (1..9), mean over 14m-160m"}
    # the confirmatory part: written once, as soon as the 36 models are done
    if os.path.exists(CONFIRMATORY_FILE):
        print(f"{CONFIRMATORY_FILE} exists; the confirmatory part is kept as it is")
    else:
        records = []
        for seed in CHECK_SEEDS:
            for size in SIZES:
                records.append(run_model(check_name(size, seed), size, seed, R_STAR[size], FULL, device, a.out))
        doc = {"provenance": provenance(records), "result": aggregate(records), "per_model": records,
               "wall_seconds": time.time() - t0}
        with open(CONFIRMATORY_FILE, "x") as f:
            json.dump(doc, f, indent=1)
        r = doc["result"]
        print("existence:", r["read_out_existence"], "| analytic:", r["read_out_analytic"],
              "| R.1 points:", r["R1_points"])
        print(f"wrote {CONFIRMATORY_FILE} ({time.time() - t0:.0f}s)", flush=True)
    # the descriptive part: 410m in fp16, its own file
    if WITH_410M and not os.path.exists(DESCRIPTIVE_410M_FILE):
        extra = [run_model(check_name(SIZE_410, seed), SIZE_410, seed, R_STAR[SIZE_410], FULL, device, a.out,
                           precision="fp16") for seed in CHECK_SEEDS]
        doc = {"provenance": provenance(extra), "descriptive_410m": extra, "wall_seconds": time.time() - t0}
        with open(DESCRIPTIVE_410M_FILE, "x") as f:
            json.dump(doc, f, indent=1)
        print(f"wrote {DESCRIPTIVE_410M_FILE} ({time.time() - t0:.0f}s)")
    return 0


def verify(path: str) -> int:
    """Locally, before a result is committed: every confirmatory record was computed by the script of the
    protocol commit; the file itself and every 410m record by this file as committed (Amendment 1)."""
    import subprocess
    here = os.path.relpath(__file__)
    show = lambda rev: subprocess.run(["git", "show", f"{rev}:{here}"], capture_output=True, check=True).stdout
    protocol = hashlib.sha256(show(PROTOCOL_COMMIT)).hexdigest()
    current = hashlib.sha256(show("HEAD")).hexdigest()
    doc = json.load(open(path))
    ok = protocol == PROTOCOL_SHA256 and doc["provenance"]["script_sha256"] == current
    for r in doc.get("per_model", []):
        ok &= r["environment"]["script_sha256"] in (protocol, current)
    for r in doc.get("descriptive_410m", []):
        ok &= r["environment"]["script_sha256"] == current
    recorded = sorted({r["environment"]["script_sha256"][:12]
                       for r in doc.get("per_model", []) + doc.get("descriptive_410m", [])})
    print(f"protocol {protocol[:12]}, current {current[:12]}; models {recorded}; file "
          f"{doc['provenance']['script_sha256'][:12]} -> {'OK' if ok else 'MISMATCH'}")
    return 0 if ok else 4


if __name__ == "__main__":
    sys.exit(main())
'''


In [ ]:
#@title 2.12 Сборка файла и проверка sha256 (обязательно)
import hashlib
EXPECTED_SHA256 = "6c0606476a6e0c1f80ec6eec2b14bd073184e1eb2b3607c1e7c5492ec85dfc8d"
missing = [i for i in range(11) if i not in PARTS]
assert not missing, f"не выполнены ячейки частей: {missing}"
data = "".join(PARTS[i] for i in range(11)).encode("utf-8")
got = hashlib.sha256(data).hexdigest()
assert got == EXPECTED_SHA256, f"sha256 {got} не совпадает с {EXPECTED_SHA256}: код изменён, прогон недействителен"
with open("pythia_address.py", "wb") as f:
    f.write(data)
print("pythia_address.py записан; sha256 совпадает:", got)


## 3. Пробный запуск

pythia-14m и pythia-14m-seed1 с крошечными настройками: проверяет загрузку, контроли и запись файлов. Числа
ничего не значат. Пишет только папку `smoke_tmp/`.

In [ ]:
#@title 3.1 Пробный запуск (сделан; включить только для повтора)
REDO_SMOKE = False  #@param {type:"boolean"}
if REDO_SMOKE:
    get_ipython().system("python pythia_address.py --smoke")
else:
    print("пропущено: пробный запуск уже сделан")


## 4. Замер времени: 160m и 410m

Время одного шага DAS при пакете 256 в fp32 и в fp16. Правило, записанное в протоколе до замера: 410m берётся
описательно в fp16, если прогноз DAS на 9 seeds не больше 3 часов; иначе убирается. Пишет
`pythia_address_time.json`.

In [ ]:
#@title 4.0 Замер (сделан; включить только для повтора)
REDO_TIME = False  #@param {type:"boolean"}
if REDO_TIME:
    get_ipython().system("python pythia_address.py --time")
else:
    print("пропущено: замер уже сделан, он в results/pythia_address_time.json репозитория")


In [ ]:
#@title 4.1 Итог замера
import json, os
if not os.path.exists("pythia_address_time.json"):
    print("файла замера нет в этой папке: замер сделан раньше, его итог записан в протоколе")
else:
    t = json.load(open("pythia_address_time.json"))
    for k, v in t.items():
        if isinstance(v, dict) and "seconds_per_step" in v:
            print(f"{k:22s} {v['seconds_per_step']:.3f} с/шаг   DAS на 9 seeds ≈ {v['das_hours_9_seeds']:.1f} ч")
    h = t.get("pythia-410m|fp16", {}).get("das_hours_9_seeds", float("inf"))
    print("видеокарта:", t["environment"]["gpu"])
    print("410m по правилу протокола:", "берётся описательно (fp16)" if h <= 3 else "убирается")


## 5. Пилот: выбор слоя и сильнейших формул

Одиннадцать пилотных моделей (seed 0 каждого размера и шесть моделей 160m с отдельными seeds данных и весов),
перебор всех слоёв на последнем токене (у 410m каждый второй), облегчённый рецепт. Правило выбора записано в
протоколе до пилота. Пишет `pythia_address_pilot.json`; готовые модели лежат в `pilot_parts/`, поэтому после обрыва
пилот продолжится с места остановки.

In [ ]:
#@title 5.0 Пилот (сделан; включить только для повтора — перезапишет файл пилота)
REDO_PILOT = False  #@param {type:"boolean"}
if REDO_PILOT:
    get_ipython().system("python pythia_address.py --pilot")
else:
    print("пропущено: пилот уже сделан, его итог записан в протоколе (часть 2.1, раздел Pilot)")


In [ ]:
#@title 5.1 Итог пилота
import json, os
if not os.path.exists("pythia_address_pilot.json"):
    print("файла пилота нет в этой папке: пилот сделан раньше, его итог записан в протоколе")
else:
    get_ipython().system("python pythia_address.py --pilot-summary pythia_address_pilot.json")


## 6. Что прислать

Два файла из папки на Диске: `pythia_address_time.json` и `pythia_address_pilot.json`. Ячейка ниже скачает их
на компьютер. После этого я впишу выбор в протокол, закоммичу его и пришлю обновлённый ноутбук для раздела 7.

In [ ]:
#@title 6.1 Скачать оба файла (уже присланы; включить только при повторе)
GET = False  #@param {type:"boolean"}
if GET:
    from google.colab import files
    for name in ("pythia_address_time.json", "pythia_address_pilot.json"):
        files.download(name)


## 7. Основной прогон

Протокол закоммичен. Две главные точки на последнем токене: последний слой и слой, где приходит число
подлежащего; диагностика на позиции подлежащего — слой 1.

36 подтверждающих моделей (seeds 1–9 × 14m, 31m, 70m, 160m) уже посчитаны и лежат в `R0/`: они возьмутся
готовыми (поправка 1 к протоколу), и подтверждающий результат `pythia_address.json` запишется сразу. Затем
410m описательно в fp16, около 3–4 ч, в свой файл `pythia_address_410m.json`. После обрыва или во второй
сессии снова выполнить разделы 1, 2 и эту ячейку: прогон продолжится со следующей модели.

In [ ]:
!python pythia_address.py --out R0


## 8. Сводка основного прогона

In [ ]:
#@title 8.1 Итог и скачивание
import json, os
from google.colab import files
if os.path.exists("pythia_address.json"):
    d = json.load(open("pythia_address.json"))
    r = d["result"]
    print("существование адреса:", r["read_out_existence"])
    print("аналитичность по точкам:", r["read_out_analytic"])
    print("точки для R.1:", r["R1_points"])
    for v, arms in r["means"].items():
        print(f"{v:13s}", " ".join(f"{k} {x['mean']:+.3f}" for k, x in arms.items()))
    files.download("pythia_address.json")
else:
    print("pythia_address.json ещё нет")
if os.path.exists("pythia_address_410m.json"):
    files.download("pythia_address_410m.json")
else:
    print("pythia_address_410m.json ещё нет (410m считается после подтверждающей части)")
